In [2]:
# assemble unified graph

import numpy as np
import torch
from torch_geometric.utils import to_undirected

sl = np.load("../data/constructs/slope_torch.npz", allow_pickle=True)
bg = np.load("../data/constructs/basin_mushgraph.npz", allow_pickle=True)
pl = np.load("../data/constructs/pit_basin_links.npz", allow_pickle=True)

N_s, N_b = sl["X"].shape[0], bg["X"].shape[0] # number slope domain, basin domain

def zscore(X):                       # nan-robust: ignore NaN for stats, then NaN->0 (col mean)
    m = torch.nanmean(X, 0)
    v = torch.nanmean((X - m)**2, 0)
    return torch.nan_to_num((X - m) / torch.sqrt(v + 1e-6))


X_s = zscore(torch.tensor(sl["X"], dtype=torch.float32))
X_b = zscore(torch.tensor(bg["X"], dtype=torch.float32))
y_s = torch.tensor(sl["y"], dtype=torch.float32)

e_slope = torch.tensor(sl["edge_index"], dtype=torch.long)
e_basin = torch.tensor(bg["edge_index"], dtype=torch.long) + N_s # bumping past slope indices
pit_t   = torch.tensor(sl["pit_torch_idx"], dtype=torch.long)
e_cross = torch.stack([pit_t[torch.tensor(pl["link_pit"])],
                       torch.tensor(pl["link_basin"]) + N_s])

edge_index = to_undirected(torch.cat([e_slope, e_basin, e_cross], 1),
                           num_nodes=N_s + N_b) # bidirectional, dedup

print(f"nodes {N_s+N_b:,} (slope {N_s:,} + basin {N_b:,}) | edges {edge_index.size(1):,} "
      f"| valid y {int(torch.isfinite(y_s).sum()):,}")

nodes 976,613 (slope 876,690 + basin 99,923) | edges 2,772,382 | valid y 876,690


In [3]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import TransformerConv

torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"

ABLATE_BASIN = False    # <-- flip to True for the matched no-basin control
Xb_in = X_b[torch.randperm(N_b)] if ABLATE_BASIN else X_b   # shuffle kills spatial basin signal, keeps structure

class HeteroGT(nn.Module):
    def __init__(self, in_s, in_b, hid=64, heads=4, layers=4):
        super().__init__()
        self.enc_s = nn.Linear(in_s, hid); self.enc_b = nn.Linear(in_b, hid)
        self.convs = nn.ModuleList(
            [TransformerConv(hid, hid // heads, heads=heads, concat=True) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hid) for _ in range(layers)])
        self.head  = nn.Linear(hid, 1)
    def forward(self, xs, xb, ei):
        h = torch.cat([self.enc_s(xs), self.enc_b(xb)], 0)
        for conv, norm in zip(self.convs, self.norms):
            h = norm(h + F.relu(conv(h, ei))) # residual + LN transformer block
        return self.head(h).squeeze(-1)

model = HeteroGT(X_s.size(1), X_b.size(1)).to(dev)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
Xs, Xb, ei, y = X_s.to(dev), Xb_in.to(dev), edge_index.to(dev), y_s.to(dev)
tr, va, te = tr.to(dev), va.to(dev), te.to(dev)



def r2(p, idx):
    t = y[idx]; return (1 - ((p[idx]-t)**2).sum() / ((t-t.mean())**2).sum()).item()

best = -1e9
for ep in range(1, 1001):
    model.train(); opt.zero_grad()
    out = model(Xs, Xb, ei); F.mse_loss(out[tr], y[tr]).backward(); opt.step
    if ep % 25 == 0:
        model.eval()
        with torch.no_grad():
            out = model(Xs, Xb, ei); vr, tr2 = r2(out, va), r2(out, te)
        best = max(best, vr)
        print(f"ep {ep:4d} | loss {F.mse_loss(out[tr], y[tr]):.4f} | val R² {vr:.3f} | test R² {tr2:.3f}")
print(f"done ({'NO-basin' if ABLATE_BASIN else 'basin'}) | best val R² {best:.3f}")     

NameError: name 'Xb_in' is not defined